# 2008年前後のジャンプの検証:CP・ROE の RMW18

`rmw18_factor_analysis.ipynb` の累積リターンで、CP と ROE の RMW18 が2008年前後に急に跳ね、その後に直線でつながる区間がありました。
また、CP と ROE だけ月数が12か月少ない(CBOP・ROA は322か月、CP・ROE は310か月)ことも分かっています。
このノートブックで、その原因を突き止めます。

0〜4章は `rmw18_factor_analysis.ipynb` と同じコードです(設定・取得関数・データ整形・期待収益性・ファクターの作成)。
検証の本体は「10. ジャンプの検証」以降です。

**調べる順番と、事前に考えられる原因**

| 段階 | 調べること | 考えられる原因 |
|---|---|---|
| 10-1 | 組成年ごとのパネルの銘柄数と、期待収益性の係数が推定できた年 | 特定の年に CP・ROE のサンプルがほぼ空になっている |
| 10-2 | 決算期ごとの自己資本の項目(株主資本合計・資本合計・純資産合計)と、自己資本成長率の件数 | **2006年の会社法施行で、自己資本の項目が「資本合計」から「株主資本合計」に切り替わり、同じ項目どうしで成長率を計算する規則のせいで、切り替わりの年に自己資本成長率がほぼ全社で欠損する**。CP・ROE だけが自己資本成長率を投資の変数に使うので、この2指標だけが影響を受ける |
| 10-3 | 2006〜2011年の月次のファクターリターンと、銘柄数・最少セルの銘柄数 | サンプルが薄い月にジャンプが起きている |
| 10-4 | 絶対値が大きい月の上位と、大型・小型、高収益・低収益への分解 | どのレッグがジャンプを生んでいるか |
| 10-5 | ジャンプした月の18セルの寄与と、最も寄与したセルの構成銘柄 | 1〜2銘柄の極端なリターンが、薄いセルを通じてファクター全体を動かしている |
| 10-6 | 寄与の大きい銘柄のリターンと時価総額の推移 | リターンや時価総額のデータの誤り |

## 0. 設定

In [ ]:
import calendar
import math
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 図の日本語表示

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

try:
    display
except NameError:  # Jupyter 以外で実行したとき
    display = print

# ===== 分析期間 =====
START_MONTH = "1995-07"              # 分析開始月(この月以降のファクターリターンで評価)
END_MONTH = None                     # 分析終了月("2024-12" など。None ならデータの最後まで)

# ===== ポートフォリオ =====
FORMATION_MONTH = 6                  # 毎年この月の月末に組成し、翌月から12か月保有
BIG_CUM_SHARE = 0.90                 # 大型:時価総額の大きい順に足して累積 90% に達するまで
BM_BPS = (0.3, 0.7)                  # B/M の分位点
PROF_BPS = (0.3, 0.7)                # 収益性の分位点
WEIGHTING = "vw"                     # "vw":時価総額加重バイアンドホールド、"ew":等ウェイトバイアンドホールド
CUM_METHOD = "index100"              # 図の累積:"index100" = 100 × Π(1 + r)、"simple" = Π(1 + r) − 1、"log" = Σ log(1 + r)
UNIVERSE_TOPIX_ONLY = True           # True:組成日時点の TOPIX 構成銘柄だけを対象にする
EXCLUDE_IND33 = {"4050", "7050", "7100", "7150", "7200"}  # 電気・ガス、銀行、証券、保険、その他金融

# ===== 収益性指標 =====
MEASURES = ["CBOP", "CP", "ROE", "ROA"]
SCALE = {"CBOP": "asset", "ROA": "asset", "CP": "equity", "ROE": "equity"}

# ===== データ =====
MISSING_SENTINEL = -999999998        # IDSQE 系の欠損値(これ以下の値はすべて欠損扱い)
CTYPE_PRIORITY = ["1", "3", "2"]     # 会計基準の優先順位(1: SEC, 3: IFRS, 2: 日本)
CTYPE_NAME = {"1": "SEC", "2": "日本", "3": "IFRS"}
ACCOUNTING_UNIT = 1e6                # 財務データの単位(百万円)→ 円に直す倍率
AVAIL_FALLBACK_MONTHS = 2            # 発表日・短信入手日が欠損のとき、期末 + 2か月で入手可能とみなす
MAX_STALENESS_MONTHS = 15            # 組成時点から見て、期末がこれより古い決算は使わない

# ===== 推定・検定 =====
WINSOR = (0.01, 0.99)
MICRO_PCT = 0.20                     # 期待収益性の推定から除くマイクロキャップ(ユニバースの時価総額 20 パーセンタイル未満)
MIN_OBS_CS = 30
NW_LAGS = 12

FF_PATH = Path("data/Japan_5_Factors.csv")
CACHE_PATH = Path("cache_raw_data.pkl")
OUTPUT_DIR = Path("outputs_rmw18")
OUTPUT_DIR.mkdir(exist_ok=True)

## 0-1. DB 接続

`temp1.ipynb` で使っている DB 接続用の関数 `fetch_data` と `get_data` を、このセルで import するか定義してください。
キャッシュ(`cache_raw_data.pkl`)がある場合は DB に接続しないので、未定義でも実行できます。

In [ ]:
# 例:from db_utils import fetch_data, get_data
# ↑ temp1.ipynb と同じ import 文・定義をここに書く

if "fetch_data" not in globals() or "get_data" not in globals():
    print("[注意] fetch_data / get_data が未定義です。キャッシュがない場合、データ取得でエラーになります。")

## 0-2. データ取得関数

`temp1.ipynb` の取得関数をそのまま載せています(`get_topix_data` の SELECT のカンマ抜けだけ修正)。
欠損値(-999999998)の置き換えは1章で行います。

In [ ]:
# PL項目の取得
def get_pl_data():
    PL_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[GPLS]         -- 売上総利益
        ,[OPINCM]       -- 営業利益
        ,[CHAGE]        -- 支払利息
        ,[PLCURREN]     -- 為替換算調整勘定，のちのちの分析で必要かもなので一応
        ,[NETINCM]      -- 当期純利益
        ,[RD]           -- 研究開発費
    FROM [IDSQE].[dbo].[NRIFIN_CON_PL4]
    """
    df = fetch_data(PL_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# BS項目の取得
def get_bs_data():
    BS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[CASH]         -- 現金・預金，のちのちの分析で必要かもなので一応
        ,[ACRC]         -- 売上債権，アクルーアルの計算要
        ,[INVENT]       -- 棚卸資産
        ,[INVSCRT]      -- 投資有価証券，別の分析でのちほど用いる
        ,[TOTALBS]      -- 資産合計
        ,[ACPY]         -- 買入債務
        ,[CAPSTTL]      -- 株主資本合計
        ,[NTASTTL]      -- 純資産合計，株主資本合計が欠損しているときに用いる
        ,[EQTY]         -- 資本合計，株主資本合計が欠損しているとき
    FROM [IDSQE].[dbo].[NRIFIN_CON_BS4]
    """
    df = fetch_data(BS_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 決算詳細
def get_accounting_details():
    DETAILS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]            -- 決算期 int yyyymm
        ,[CTYPE]            -- 会計基準 1: SEC式 2: 日本式 3: IFRS式
        ,[SETTLEMENT_FLG]   -- 本中区分 1: 本決算 2: 中間決算 3: 第1四半期決算 4: 第3四半期決算
        ,[FDATE]            -- 決算年月日 char yyyymmdd
        ,[NMONTH]           -- 決算月数 通常12
        ,[ANNOUNCE]         -- 決算発表日，[RCVD]より欠損(-999999998)が多い
        ,[RCVD]             -- 決算短信入手日
    FROM [IDSQE].[dbo].[NRIFIN_CON_AT4]
    """
    df = fetch_data(DETAILS_QUERY, 'idsqe')
    # 欠損値は-999999998
    # 決算発表、短信入手日どちらも欠損している場合は、決算期プラス2か月する（日本の法律）
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 会社属性
# NRI_CODE -> 証券コード
def get_nri_to_ticker_table():
    TABLE_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,[ITEM_DATA]        -- 証券コード
    FROM [IDSQE].[dbo].[COMPANY_CODE]
    WHERE ITEM = 'OLDCD'            -- 証券コード
    """
    df = fetch_data(TABLE_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 業種コード
def get_industry():
    IND_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,c.[ITEM_DATA]
        ,n.[IND_NAME]
    FROM [IDSQE].[dbo].[COMPANY_INDUSTRY_CLASS] c
    JOIN [IDSQE].[dbo].[INDUSTRY_CLASS_NAME] n ON c.ITEM = n.IND
    WHERE [ITEM] = 'IND33'   -- 東証33業種
    """
    df = fetch_data(IND_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# TOPIX構成銘柄
def get_topix_data():
    query = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]
        ,[TO_DATE]
        ,[ITEM]
        ,[ITEM_DATA]
    FROM [IDSQE].[dbo].[STOCK_ISSUE_INDEX]
    WHERE [ITEM] = 'TOPIX';
    """
    return get_data(query, "idsqe")


# 時価総額の取得
def get_market_cap_data():
    # 月次時価総額と、各月末時点で有効な証券コードを取得
    query = """
    SELECT
        prc.[DATE] AS [date],
        prc.[CAP] AS market_cap,
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_PRC] AS prc
    CROSS APPLY (
        -- YYYYMMから月末日（YYYYMMDD）を作成
        SELECT CONVERT(
            int,
            CONVERT(
                char(8),
                EOMONTH(DATEFROMPARTS(
                    prc.[DATE] / 100,
                    prc.[DATE] % 100,
                    1
                )),
                112
            )
        ) AS month_end
    ) AS d
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON prc.[BID] = id.[BID]
        AND id.[IDTYPE] = 'LOCALID'
        AND id.[FDATE] <= d.[month_end]
        AND (id.[TDATE] >= d.[month_end] OR id.[TDATE] IS NULL);
    """
    df = get_data(query, "riskmodels")

    # 証券コードと数値列を整形
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    df["date"] = pd.to_numeric(df["date"], errors="raise").astype(int)
    df["market_cap"] = (
        pd.to_numeric(df["market_cap"], errors="raise")
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
    )
    # 月次Period列を追加
    df["month"] = pd.to_datetime(df["date"].astype(str), format="%Y%m").dt.to_period("M")
    return df


# 銘柄リターンの取得
def get_return_data():
    # 月次リターンと、各月初時点で有効な証券コードを取得
    query = """
    SELECT
        rtn.[DATE],
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code,
        rtn.[RTN]
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_RTN] AS rtn
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON rtn.[BID] = id.[BID]
    WHERE id.[IDTYPE] = 'LOCALID'
      AND TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
          >= TRY_CONVERT(date, CONVERT(varchar(8), id.[FDATE]), 112)
      AND (
          id.[TDATE] IS NULL
          OR TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
             <= TRY_CONVERT(date, CONVERT(varchar(8), id.[TDATE]), 112)
      );
    """
    df = get_data(query, "riskmodels")

    # 証券コード・年月を整形し、リターンを%から小数へ変換
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    dates = pd.to_numeric(df["DATE"], errors="raise").astype(int)
    df["month"] = pd.to_datetime(dates.astype(str), format="%Y%m").dt.to_period("M")
    df["ret"] = pd.to_numeric(df["RTN"], errors="raise").astype(float) / 100
    return df[["security_code", "month", "ret"]].copy()

## 0-3. データ読み込み

上の取得関数で DB からデータを取得し、キャッシュ(`cache_raw_data.pkl`)に保存します。
2回目以降はキャッシュを読みます。取り直したいときは `USE_CACHE = False` にしてください。

In [ ]:
USE_CACHE = True


def load_raw_data(use_cache=USE_CACHE):
    if "取得データ" in globals():  # temp1.ipynb の取得結果がカーネルに残っている場合
        print("既存の `取得データ` を使います")
        return globals()["取得データ"]
    if use_cache and CACHE_PATH.exists():
        print(f"キャッシュ {CACHE_PATH} を読み込みます")
        with open(CACHE_PATH, "rb") as f:
            return pickle.load(f)
    funcs = {
        "PL": get_pl_data, "BS": get_bs_data, "決算詳細": get_accounting_details,
        "証券コード対応表": get_nri_to_ticker_table, "業種": get_industry,
        "TOPIX構成銘柄": get_topix_data, "時価総額": get_market_cap_data, "月次リターン": get_return_data,
    }
    data = {}
    for name, fn in funcs.items():
        print(f"取得中:{name}")
        data[name] = fn()
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(data, f)
    return data


raw = load_raw_data()
for k, v in raw.items():
    print(f"{k}: {v.shape}")

### 共通の関数

In [ ]:
def clean_idsqe(df):
    """IDSQE 系テーブルの整形:欠損値(-999999998 以下)を NaN に、文字列の前後空白を除去"""
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c]):
            df[c] = df[c].mask(df[c] <= MISSING_SENTINEL)
        elif pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df


def norm_code(s):
    """'2'・2・2.0 のどれで来ても '2' という文字列にそろえる"""
    num = pd.to_numeric(s, errors="coerce")
    out = s.astype(str).str.strip()
    return out.where(num.isna(), num.round().astype("Int64").astype(str))


def norm_sec(s):
    return s.astype(str).str.strip().str.upper()


def to_date(s):
    """yyyymmdd の整数 → Timestamp(0・99999999・欠損は NaT)"""
    v = pd.to_numeric(s, errors="coerce")
    v = v.where((v > 19000000) & (v < 21000000))
    return pd.to_datetime(v.astype("Int64").astype(str), format="%Y%m%d", errors="coerce")


def to_midx(s):
    """月(Period・'YYYY-MM'・yyyymm 整数)→ 通し月番号 year*12 + month - 1"""
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype(int)
        return (v // 100) * 12 + v % 100 - 1
    dt = pd.to_datetime(s.astype(str))
    return dt.dt.year * 12 + dt.dt.month - 1


def month_str_to_midx(m):
    ts = pd.Timestamp(m)
    return ts.year * 12 + ts.month - 1


def midx_to_ts(m):
    m = np.asarray(m, dtype=int)
    return pd.to_datetime({"year": m // 12, "month": m % 12 + 1, "day": 1}) + pd.offsets.MonthEnd(0)


def formation_date(y):
    return pd.Timestamp(int(y), FORMATION_MONTH, calendar.monthrange(int(y), FORMATION_MONTH)[1])


def formation_int(y):
    d = formation_date(y)
    return d.year * 10000 + d.month * 100 + d.day


def winsorize_by(df, cols, by, limits=WINSOR):
    df = df.copy()
    for c in cols:
        g = df.groupby(by)[c]
        lo = g.transform(lambda x: x.quantile(limits[0]))
        hi = g.transform(lambda x: x.quantile(limits[1]))
        df[c] = df[c].clip(lo, hi)
    return df


def ols_nw(y, X, lags=NW_LAGS):
    """OLS(定数項あり)と Newey-West の t 値。X が (T, 0) なら平均の検定"""
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float).reshape(len(y), -1)
    X = np.column_stack([np.ones(len(y)), X])
    ok = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
    y, X = y[ok], X[ok]
    T = len(y)
    if T <= X.shape[1] + 1:
        return np.full(X.shape[1], np.nan), np.full(X.shape[1], np.nan), T
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ y
    u = y - X @ b
    Xu = X * u[:, None]
    S = Xu.T @ Xu
    for l in range(1, min(lags, T - 1) + 1):
        w = 1 - l / (lags + 1)
        G = Xu[l:].T @ Xu[:-l]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    return b, b / np.sqrt(np.diag(V)), T


def mean_t(x, lags=NW_LAGS):
    x = pd.Series(x).dropna()
    b, t, T = ols_nw(x.values, np.empty((len(x), 0)), lags)
    return b[0], t[0], T


def p_one_sided(t):
    """平均 > 0 の片側 p 値(正規近似)"""
    return 0.5 * math.erfc(t / math.sqrt(2)) if pd.notna(t) else np.nan


def p_two_sided(t):
    return math.erfc(abs(t) / math.sqrt(2)) if pd.notna(t) else np.nan


def ols_cs(y, X):
    Xc = np.column_stack([np.ones(len(y)), X])
    b, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    return b


def asof_lookup(keys, table, value_cols, date_col):
    """NRI_CODE と日付(yyyymmdd 整数)で、FROM_DATE ≤ 日付 ≤ TO_DATE の行を引く"""
    k = keys[["NRI_CODE", date_col]].drop_duplicates()
    m = k.merge(table, on="NRI_CODE", how="inner")
    m = m[(m["FROM_DATE"] <= m[date_col]) & (m[date_col] <= m["TO_DATE"])]
    m = m.sort_values("FROM_DATE").drop_duplicates(["NRI_CODE", date_col], keep="last")
    return m[["NRI_CODE", date_col] + value_cols]


START_MIDX = month_str_to_midx(START_MONTH)
END_MIDX = month_str_to_midx(END_MONTH) if END_MONTH else None
print(f"分析期間:{START_MONTH} 〜 {END_MONTH or 'データの最後'}")

## 1. 財務データの整形と収益性指標の計算

1. 欠損値を NaN に置き換え、**本決算・決算月数12** の行だけを残す
2. PL・BS・決算詳細を (NRI_CODE, 決算期, 会計基準) で結合(会計基準ごとに別の行のまま)
3. 前期・翌期の値は (NRI_CODE, 会計基準, 決算期 ± 12か月) で結合 → 分子と分母の会計基準が一致
4. 入手可能日から、その決算を使う組成年(6月末までに公表されていればその年)を決める

In [ ]:
KEY = ["NRI_CODE", "FTERM", "CTYPE"]

pl = clean_idsqe(raw["PL"])
bs = clean_idsqe(raw["BS"])
at = clean_idsqe(raw["決算詳細"])
for d in (pl, bs, at):
    d["NRI_CODE"] = norm_sec(d["NRI_CODE"])
    d["CTYPE"] = norm_code(d["CTYPE"])
    d["FTERM"] = pd.to_numeric(d["FTERM"]).astype(int)
at["SETTLEMENT_FLG"] = norm_code(at["SETTLEMENT_FLG"])
at = at[(at["SETTLEMENT_FLG"] == "1") & (at["NMONTH"] == 12)]


def dedupe(df, name):
    n = df.duplicated(KEY).sum()
    if n:
        print(f"[注意] {name}: (NRI_CODE, 決算期, 会計基準) の重複が {n} 行 → 先頭の行を残します")
        df = df.drop_duplicates(KEY)
    return df


at, pl, bs = dedupe(at, "決算詳細"), dedupe(pl, "PL"), dedupe(bs, "BS")
acct = at.merge(pl, on=KEY, how="inner").merge(bs, on=KEY, how="inner")
print(f"本決算・12か月の財務データ:{len(acct):,} 行")

# 決算期、期末日、入手可能日
acct["fy_idx"] = (acct["FTERM"] // 100) * 12 + acct["FTERM"] % 100 - 1
fterm_end = midx_to_ts(acct["fy_idx"].to_numpy())
acct["fdate"] = to_date(acct["FDATE"]).fillna(pd.Series(fterm_end.values, index=acct.index))
ann = to_date(acct["ANNOUNCE"])
rcv = to_date(acct["RCVD"])
fallback = acct["fdate"] + pd.offsets.MonthEnd(AVAIL_FALLBACK_MONTHS)
acct["avail_date"] = ann.where(ann >= acct["fdate"]).fillna(rcv.where(rcv >= acct["fdate"])).fillna(fallback)

# 自己資本・総資産
acct["TA"] = acct["TOTALBS"].where(acct["TOTALBS"] > 0)
be_raw = acct["CAPSTTL"].fillna(acct["EQTY"]).fillna(acct["NTASTTL"])
acct["BE"] = be_raw.where(be_raw > 0)

# 前期(同じ会計基準)の値
PREV_COLS = ["TA", "BE", "CAPSTTL", "EQTY", "NTASTTL", "ACRC", "INVENT", "ACPY"]
prev = acct[["NRI_CODE", "CTYPE", "fy_idx"] + PREV_COLS].copy()
prev["fy_idx"] += 12
prev["prev_exists"] = True
acct = acct.merge(prev.rename(columns={c: c + "_p" for c in PREV_COLS}),
                  on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
acct["prev_exists"] = acct["prev_exists"].eq(True)


def delta_item(cur, prv, exists):
    d = cur - prv
    d = d.mask(cur.isna() & prv.isna(), 0.0)
    return d.where(exists)


acct["accrual"] = (delta_item(acct["ACRC"], acct["ACRC_p"], acct["prev_exists"])
                   + delta_item(acct["INVENT"], acct["INVENT_p"], acct["prev_exists"])
                   - delta_item(acct["ACPY"], acct["ACPY_p"], acct["prev_exists"]))
int_exp = acct["CHAGE"].fillna(0.0)
rd = acct["RD"].fillna(0.0)
numer = {
    "CBOP": acct["OPINCM"] + rd - acct["accrual"],
    "CP": acct["OPINCM"] - int_exp - acct["accrual"],
    "ROE": acct["NETINCM"],
    "ROA": acct["OPINCM"],
}
for m in MEASURES:
    denom = acct["TA_p"] if SCALE[m] == "asset" else acct["BE_p"]
    acct[m] = numer[m] / denom.where(denom > 0)

# 成長率(自己資本は両期で同じ項目がそろうものだけ)
acct["g_TA"] = acct["TA"] / acct["TA_p"] - 1
g_be = pd.Series(np.nan, index=acct.index)
for col in ["CAPSTTL", "EQTY", "NTASTTL"]:
    ok = (acct[col] > 0) & (acct[col + "_p"] > 0) & g_be.isna()
    g_be[ok] = acct.loc[ok, col] / acct.loc[ok, col + "_p"] - 1
acct["g_BE"] = g_be

# 1期前と1期後(同じ会計基準)
LAG_COLS = MEASURES + ["g_TA", "g_BE"]
lag = acct[["NRI_CODE", "CTYPE", "fy_idx"] + LAG_COLS].copy()
lag["fy_idx"] += 12
acct = acct.merge(lag.rename(columns={c: c + "_lag" for c in LAG_COLS}), on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
lead = acct[["NRI_CODE", "CTYPE", "fy_idx", "avail_date"] + MEASURES].copy()
lead["fy_idx"] -= 12
acct = acct.merge(lead.rename(columns={c: c + "_lead" for c in MEASURES + ["avail_date"]}),
                  on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
for m in MEASURES:
    acct["d" + m] = acct[m] - acct[m + "_lag"]

# 組成年:入手可能日が組成月末までなら、その年の組成で使える
acct = acct[acct["avail_date"].notna()].copy()
av = acct["avail_date"]
acct["form_year"] = (av.dt.year + (av.dt.month > FORMATION_MONTH).astype(int)).astype(int)
acct["stale_ok"] = (acct["form_year"] * 12 + FORMATION_MONTH - 1 - acct["fy_idx"]) <= MAX_STALENESS_MONTHS
acct["prio"] = acct["CTYPE"].map({c: i for i, c in enumerate(CTYPE_PRIORITY)}).fillna(99)

print("指標ごとの計算できた件数(会計基準別)")
display(acct.groupby(acct["CTYPE"].map(CTYPE_NAME).fillna(acct["CTYPE"]))[MEASURES].count().T)

## 2. 組成年への割り当て・会計基準の選択・市場データとの結合

- 各企業・各組成年について、使える決算のうち最新の決算期を選び、その中で必要な変数(収益性・その変化・投資・自己資本)が
  **同じ会計基準ですべて計算できる** 行を、会計基準の優先順位で1つ選ぶ(指標ごと)
- 組成日(6月末)時点の証券コード・業種・TOPIX 構成を引き、6月末の時価総額と決算期末の時価総額を付ける
- ユニバース:TOPIX 構成銘柄(`UNIVERSE_TOPIX_ONLY`)のうち、電気・ガス・金融を除く銘柄

In [ ]:
code_tbl = clean_idsqe(raw["証券コード対応表"])
code_tbl["NRI_CODE"] = norm_sec(code_tbl["NRI_CODE"])
code_tbl["sec"] = norm_sec(code_tbl["ITEM_DATA"])
code_tbl["FROM_DATE"] = code_tbl["FROM_DATE"].fillna(0)
code_tbl["TO_DATE"] = code_tbl["TO_DATE"].fillna(99999999)

ind_tbl = clean_idsqe(raw["業種"])
ind_tbl["NRI_CODE"] = norm_sec(ind_tbl["NRI_CODE"])
ind_tbl["ind33"] = norm_code(ind_tbl["ITEM_DATA"]).str.zfill(4)
ind_tbl = ind_tbl[["NRI_CODE", "FROM_DATE", "TO_DATE", "ind33"]].drop_duplicates()
ind_tbl["FROM_DATE"] = ind_tbl["FROM_DATE"].fillna(0)
ind_tbl["TO_DATE"] = ind_tbl["TO_DATE"].fillna(99999999)

topix_tbl = clean_idsqe(raw["TOPIX構成銘柄"])
topix_tbl["NRI_CODE"] = norm_sec(topix_tbl["NRI_CODE"])
topix_tbl = topix_tbl[["NRI_CODE", "FROM_DATE", "TO_DATE"]].drop_duplicates()
topix_tbl["FROM_DATE"] = topix_tbl["FROM_DATE"].fillna(0)
topix_tbl["TO_DATE"] = topix_tbl["TO_DATE"].fillna(99999999)
topix_tbl["topix"] = True

mcap = raw["時価総額"].copy()
mcap["sec"] = norm_sec(mcap["security_code"])
mcap["midx"] = to_midx(mcap["month"]) if "month" in mcap else to_midx(mcap["date"])
mcap["me"] = pd.to_numeric(mcap["market_cap"], errors="coerce")
mcap = mcap.loc[mcap["me"] > 0, ["sec", "midx", "me"]].drop_duplicates(["sec", "midx"])

ret = raw["月次リターン"].copy()
ret["sec"] = norm_sec(ret["security_code"])
ret["midx"] = to_midx(ret["month"])
ret["ret"] = pd.to_numeric(ret["ret"], errors="coerce")
ret = ret.loc[ret["ret"].notna(), ["sec", "midx", "ret"]].drop_duplicates(["sec", "midx"])


def select_records(req_cols):
    d = acct[acct["stale_ok"]].dropna(subset=req_cols)
    d = d.sort_values(["NRI_CODE", "form_year", "fy_idx", "prio"], ascending=[True, True, False, True])
    return d.drop_duplicates(["NRI_CODE", "form_year"], keep="first").copy()


def attach_market(d):
    d = d.copy()
    d["F_int"] = d["form_year"].map(formation_int)
    d = d.merge(asof_lookup(d, code_tbl, ["sec"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d = d.merge(asof_lookup(d, ind_tbl, ["ind33"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d = d.merge(asof_lookup(d, topix_tbl, ["topix"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d["topix"] = d["topix"].eq(True)
    d["midx_form"] = d["form_year"] * 12 + FORMATION_MONTH - 1
    d = d.merge(mcap.rename(columns={"midx": "midx_form", "me": "ME_form"}), on=["sec", "midx_form"], how="left")
    d = d.merge(mcap.rename(columns={"midx": "fy_idx", "me": "ME_fye"}), on=["sec", "fy_idx"], how="left")
    d = d[d["sec"].notna() & (d["ME_form"] > 0) & d["ind33"].notna() & ~d["ind33"].isin(EXCLUDE_IND33)].copy()
    if UNIVERSE_TOPIX_ONLY:
        d = d[d["topix"]].copy()
    d["lnME"] = np.log(d["ME_form"])
    bm = d["BE"] * ACCOUNTING_UNIT / d["ME_fye"]
    d["lnBM"] = np.log(bm.where(bm > 0))
    d["micro"] = d["ME_form"] < d.groupby("form_year")["ME_form"].transform(lambda s: s.quantile(MICRO_PCT))
    return d.drop_duplicates(["NRI_CODE", "form_year"])


def build_measure_panel(m):
    inv_col = "g_TA_lag" if SCALE[m] == "asset" else "g_BE_lag"
    d = select_records([m, "d" + m, inv_col, "BE"])
    d = d.rename(columns={m: "Y", "d" + m: "dY", m + "_lead": "Y_lead",
                          "avail_date_lead": "avail_lead", inv_col: "inv"})
    keep = ["NRI_CODE", "form_year", "CTYPE", "fy_idx", "BE", "Y", "dY", "Y_lead", "avail_lead", "inv"]
    d = attach_market(d[keep]).dropna(subset=["lnBM"])
    return winsorize_by(d, ["Y", "dY", "inv", "lnBM"], "form_year")


panels = {}
for m in MEASURES:
    panels[m] = build_measure_panel(m)
    p = panels[m]
    print(f"{m}: {len(p):,} 企業年、組成年 {p['form_year'].min()}〜{p['form_year'].max()}、"
          f"会計基準 {p['CTYPE'].map(CTYPE_NAME).value_counts().to_dict()}")

## 3. 期待収益性の推定

組成年 y の期待収益性 Yhat は、前年のコホート(y−1 年組成)のうち **y 年6月末までに翌期の実績が公表された企業** で
翌期の収益性を回帰し、その係数を y 年の説明変数に当てはめて作ります(先読みなし)。

下の表は、年ごとの回帰の係数の時系列平均と Newey-West t 値(ラグ3)、決定係数 R²・自由度調整済み R² の平均、平均銘柄数です。

In [ ]:
XCOLS = ["Y", "dY", "lnBM", "lnME", "inv"]
NW_LAGS_ANNUAL = 3


def add_expected(p):
    p = p.copy()
    coef_rows = {}
    for y in sorted(p["form_year"].unique()):
        tr = p[(p["form_year"] == y - 1) & ~p["micro"] & (p["avail_lead"] <= formation_date(y))]
        tr = tr.dropna(subset=XCOLS + ["Y_lead"])
        if len(tr) < MIN_OBS_CS:
            continue
        lo, hi = tr["Y_lead"].quantile(WINSOR)
        yv = tr["Y_lead"].clip(lo, hi).to_numpy(float)
        X = tr[XCOLS].to_numpy(float)
        b = ols_cs(yv, X)
        e = yv - (b[0] + X @ b[1:])
        n, k = len(yv), len(XCOLS) + 1
        r2 = 1 - (e @ e) / ((yv - yv.mean()) @ (yv - yv.mean()))
        coef_rows[y] = np.r_[b, r2, 1 - (1 - r2) * (n - 1) / (n - k), n]
    if not coef_rows:
        raise ValueError("期待収益性を推定できる年がありません")
    coef = pd.DataFrame.from_dict(coef_rows, orient="index",
                                  columns=["const"] + XCOLS + ["R2", "adjR2", "N"]).sort_index()
    p["Yhat"] = np.nan
    for y, b in coef.iterrows():
        mask = p["form_year"] == y
        p.loc[mask, "Yhat"] = b["const"] + p.loc[mask, XCOLS].to_numpy(float) @ b[XCOLS].to_numpy(float)
    return p, coef


first_stage_coef = {}
for m in MEASURES:
    panels[m], first_stage_coef[m] = add_expected(panels[m])

fs_rows = {}
for m, c in first_stage_coef.items():
    r = {}
    for v in ["const"] + XCOLS:
        mu, t, _ = mean_t(c[v], NW_LAGS_ANNUAL)
        r[(v, "係数")] = mu
        r[(v, "t値")] = t
    r[("決定係数", "R2")] = c["R2"].mean()
    r[("決定係数", "adjR2")] = c["adjR2"].mean()
    r[("", "平均N")] = c["N"].mean()
    r[("", "年数")] = len(c)
    fs_rows[m] = r
first_stage_summary = pd.DataFrame(fs_rows).T
print("期待収益性の推定:年次回帰の係数の平均・t 値と決定係数")
display(first_stage_summary)

## 4. ポートフォリオの作成

### 4-1. 月次リターン(バイアンドホールドのウェイト)

組成年 y の保有期間は y 年7月〜y+1 年6月。各銘柄のウェイトは

- 時価総額加重(`"vw"`):6月末の時価総額 × 7月から前月末までの累積リターン
- 等ウェイト(`"ew"`):1 × 7月から前月末までの累積リターン(7月初めに等金額で買って保有)

In [ ]:
secs = set()
for p in panels.values():
    secs |= set(p["sec"])
mlong = ret[ret["sec"].isin(secs)].copy()
mlong["form_year"] = (mlong["midx"] - FORMATION_MONTH) // 12
mlong = mlong.sort_values(["sec", "midx"])
gross = (1 + mlong["ret"].clip(lower=-0.99)).groupby([mlong["sec"], mlong["form_year"]]).cumprod()
mlong["growth_before"] = gross.groupby([mlong["sec"], mlong["form_year"]]).shift(1).fillna(1.0)
mlong = mlong[mlong["midx"] >= START_MIDX]
if END_MIDX is not None:
    mlong = mlong[mlong["midx"] <= END_MIDX]
print(f"月次リターン:{len(mlong):,} 行、{mlong['midx'].nunique()} か月、加重:{WEIGHTING}")

### 4-2. ソートとファクター

- `assign_groups`:規模 → B/M(規模の中で)→ 収益性(規模 × B/M の中で)の従属ソート。`bm_adjust=False` なら B/M を飛ばす
- `make_rmw`:18(または6)ポートフォリオのリターンから、RMW = ½(大型 高 − 低)+ ½(小型 高 − 低)

In [ ]:
def big_threshold(me, share=BIG_CUM_SHARE):
    """時価総額の大きい順に足して累積 share に達するまでを大型とするときの、大型の最小時価総額"""
    s = np.sort(np.asarray(me, dtype=float))[::-1]
    cum_before = np.cumsum(s) - s
    return s[cum_before / s.sum() < share].min()


def tercile(d, col, keys, bps):
    lo = d.groupby(keys)[col].transform(lambda x: x.quantile(bps[0]))
    hi = d.groupby(keys)[col].transform(lambda x: x.quantile(bps[1]))
    return pd.Series(np.where(d[col] <= lo, "L", np.where(d[col] >= hi, "H", "M")), index=d.index)


def assign_groups(df, signal, bm_adjust=True):
    d = df.dropna(subset=[signal, "lnBM", "ME_form"]).copy()
    thr = d.groupby("form_year")["ME_form"].transform(big_threshold)
    d["size"] = np.where(d["ME_form"] >= thr, "B", "S")
    if bm_adjust:
        d["bm"] = tercile(d, "lnBM", ["form_year", "size"], BM_BPS)
        keys = ["form_year", "size", "bm"]
    else:
        d["bm"] = "A"
        keys = ["form_year", "size"]
    d["prof"] = tercile(d, signal, keys, PROF_BPS)
    d["grp"] = d["size"] + "/" + d["bm"] + "/" + d["prof"]
    return d


def portfolio_returns(hold):
    d = hold[["sec", "form_year", "ME_form", "grp"]].merge(
        mlong[["sec", "form_year", "midx", "ret", "growth_before"]], on=["sec", "form_year"])
    if WEIGHTING == "vw":
        d["w"] = d["ME_form"] * d["growth_before"]
    elif WEIGHTING == "ew":
        d["w"] = d["growth_before"]
    else:
        raise ValueError("WEIGHTING は 'vw' か 'ew'")
    d = d[(d["w"] > 0) & d["ret"].notna()]
    d["wr"] = d["ret"] * d["w"]
    g = d.groupby(["midx", "grp"])
    return (g["wr"].sum() / g["w"].sum()).unstack("grp"), g.size().unstack("grp")


def make_rmw(df, signal, bm_adjust=True):
    h = assign_groups(df, signal, bm_adjust)
    rets, counts = portfolio_returns(h)
    bms = ["L", "M", "H"] if bm_adjust else ["A"]
    cells = [f"{s}/{b}/{q}" for s in ["B", "S"] for b in bms for q in ["L", "M", "H"]]
    rets = rets.reindex(columns=cells)
    counts = counts.reindex(columns=cells)

    def leg(size, prof):  # B/M グループの単純平均
        return rets[[f"{size}/{b}/{prof}" for b in bms]].mean(axis=1, skipna=False)

    big = leg("B", "H") - leg("B", "L")
    small = leg("S", "H") - leg("S", "L")
    groups = h[["NRI_CODE", "form_year", "size", "bm", "prof", "ME_form", "Y", "Yhat"]]
    return {"factor": 0.5 * big + 0.5 * small, "big": big, "small": small,
            "rets": rets, "counts": counts, "groups": groups}

In [ ]:
SIGNALS = {"実績": "Y", "期待": "Yhat"}
FACTOR_TYPES = {"RMW18": True, "RMW6": False}   # True:B/M を調整する
factors = {}
for m in MEASURES:
    # 実績と期待で同じ銘柄を使う(両方がそろう銘柄だけ)
    p = panels[m].dropna(subset=["Y", "Yhat"])
    for sname, col in SIGNALS.items():
        for fname, adj in FACTOR_TYPES.items():
            factors[(m, sname, fname)] = make_rmw(p, col, adj)
print(f"{len(factors)} 本のファクターを作成しました")

# セルごとの平均銘柄数(例:CBOP・実績・RMW18)
cnt = factors[("CBOP", "実績", "RMW18")]["counts"].mean()
print("CBOP・実績・RMW18 の18ポートフォリオの平均銘柄数(規模/B/M/収益性)")
display(cnt.to_frame("平均銘柄数").T)

## 10. ジャンプの検証

### 10-0. 検証の設定

In [ ]:
DIAG_MEASURES = ["CP", "ROE", "CBOP", "ROA"]   # CBOP・ROA は比較用(総資産成長率を使う指標)
WINDOW = ("2006-07", "2011-06")               # 月次で詳しく見る期間
YEAR_RANGE = (2003, 2012)                     # 組成年・決算期の表で見る範囲
JUMP_MEASURE = "CP"                           # 10-5・10-6 で深掘りする指標
JUMP_SIGNAL = "実績"                          # "実績" か "期待"
JUMP_MONTH = None                             # 深掘りする月("2008-10" など)。None なら絶対値が最大の月
N_TOP = 10


def mlabel(i):
    return midx_to_ts([int(i)])[0].strftime("%Y-%m")


W0, W1 = month_str_to_midx(WINDOW[0]), month_str_to_midx(WINDOW[1])

### 10-1. 組成年ごとのサンプルの大きさ

- **候補の決算(投資なし / 投資あり)**:その組成年に使える本決算のうち、収益性・その変化・自己資本がそろうもの(投資の条件の前後)。
  市場データやユニバースで絞る前の件数です。ここで「投資あり」だけが急に減っていれば、投資の変数が原因です
- **パネル**:ユニバースで絞った後の銘柄数、**Yhat あり**:期待収益性まで計算できた銘柄数(= ファクターに使う銘柄)
- **係数あり**:その組成年に期待収益性の係数を推定できたか(1 = できた)

In [ ]:
rows = {}
for m in DIAG_MEASURES:
    inv_col = "g_TA_lag" if SCALE[m] == "asset" else "g_BE_lag"
    base_ok = acct["stale_ok"] & acct[[m, "d" + m, "BE"]].notna().all(axis=1)
    rows[(m, "候補(投資なし)")] = acct[base_ok].drop_duplicates(["NRI_CODE", "form_year"]).groupby("form_year").size()
    rows[(m, "候補(投資あり)")] = acct[base_ok & acct[inv_col].notna()].drop_duplicates(["NRI_CODE", "form_year"]).groupby("form_year").size()
    rows[(m, "パネル")] = panels[m].groupby("form_year").size()
    rows[(m, "Yhat あり")] = panels[m].dropna(subset=["Y", "Yhat"]).groupby("form_year").size()
    rows[(m, "係数あり")] = pd.Series(1, index=first_stage_coef[m].index)
sample_by_year = pd.DataFrame(rows).fillna(0).astype(int)
display(sample_by_year.loc[YEAR_RANGE[0]:YEAR_RANGE[1]])

### 10-2. 自己資本の項目と成長率(決算期の年ごと)

自己資本の成長率は「両期で同じ項目がそろうもの」だけで計算しています(株主資本合計どうし、資本合計どうし、純資産合計どうし)。
2006年5月の会社法施行で、項目が「資本合計」から「株主資本合計・純資産合計」に切り替わった決算期では、前期と同じ項目がそろわず、
成長率がほとんど計算できない可能性があります。総資産成長率(`g_TA`)と比べてください。

投資の変数は1期前の成長率(`g_BE_lag`)なので、成長率が欠損した決算期の **翌期** の決算を使う組成年で、CP・ROE のサンプルが消えます。

In [ ]:
fy_year = acct["fdate"].dt.year
items = ["CAPSTTL", "EQTY", "NTASTTL", "g_BE", "g_TA", "g_BE_lag", "g_TA_lag"]
equity_items = acct[items].notna().groupby(fy_year).sum()
equity_items.insert(0, "本決算の件数", acct.groupby(fy_year).size())
equity_items.index.name = "決算期末の年"
display(equity_items.loc[YEAR_RANGE[0]:YEAR_RANGE[1]])

### 10-3. 月次のファクターリターンと銘柄数(2006〜2011年)

RMW18 の月次リターン(%)と、その月にポートフォリオに入っていた銘柄数の合計・最少セルの銘柄数です。
リターンが空欄の月は、どこかのセルに銘柄がいないなどでファクターを計算できなかった月で、累積リターンの図では直線でつながります。

In [ ]:
ret_rows, n_rows, min_rows = {}, {}, {}
for m in DIAG_MEASURES:
    for s in SIGNALS:
        fac = factors[(m, s, "RMW18")]
        idx = range(W0, W1 + 1)
        ret_rows[(m, s)] = fac["factor"].reindex(idx) * 100
        cnt = fac["counts"].reindex(idx)
        n_rows[(m, s)] = cnt.sum(axis=1, min_count=1)
        min_rows[(m, s)] = cnt.min(axis=1)
window_table = pd.concat({"RMW18(%)": pd.DataFrame(ret_rows), "銘柄数": pd.DataFrame(n_rows),
                          "最少セル": pd.DataFrame(min_rows)}, axis=1)
window_table.index = [mlabel(i) for i in window_table.index]
pd.set_option("display.max_rows", 200)
display(window_table)

### 10-4. 絶対値が大きい月の上位と、その分解

全期間で RMW18 の絶対値が大きい月を、指標・実績/期待ごとに上位 `N_TOP` 個表示します。

- 大型レッグ = 大型の(高収益 − 低収益)、小型レッグ = 小型の(高収益 − 低収益)。RMW18 = ½(大型レッグ + 小型レッグ)
- 高収益・低収益の列は、規模ごとに3つの B/M グループのリターンを平均したもの

In [ ]:
def legs_table(fac, months):
    rets = fac["rets"]
    out = {}
    for size, lab in [("B", "大型"), ("S", "小型")]:
        hi = rets[[f"{size}/{b}/H" for b in "LMH"]].mean(axis=1, skipna=False)
        lo = rets[[f"{size}/{b}/L" for b in "LMH"]].mean(axis=1, skipna=False)
        out[f"{lab} 高収益(%)"] = hi.reindex(months) * 100
        out[f"{lab} 低収益(%)"] = lo.reindex(months) * 100
        out[f"{lab}レッグ(%)"] = (hi - lo).reindex(months) * 100
    out["RMW18(%)"] = fac["factor"].reindex(months) * 100
    out["銘柄数"] = fac["counts"].reindex(months).sum(axis=1)
    out["最少セル"] = fac["counts"].reindex(months).min(axis=1)
    t = pd.DataFrame(out)
    t.index = [mlabel(i) for i in t.index]
    return t


for m in ["CP", "ROE"]:
    for s in SIGNALS:
        fac = factors[(m, s, "RMW18")]
        top = fac["factor"].abs().dropna().nlargest(N_TOP).index
        print(f"■ {m}・{s}:RMW18 の絶対値が大きい月")
        display(legs_table(fac, top))

### 10-5. ジャンプした月の18セルの寄与と、最も寄与したセルの構成銘柄

RMW18 = Σ(セルの係数 × セルのリターン)で、係数は高収益のセルが +1/6、低収益のセルが −1/6、中収益のセルは 0 です。
寄与が最も大きいセルについて、構成銘柄のウェイト・リターン・寄与を表示します。

In [ ]:
fac = factors[(JUMP_MEASURE, JUMP_SIGNAL, "RMW18")]
jm = month_str_to_midx(JUMP_MONTH) if JUMP_MONTH else int(fac["factor"].abs().idxmax())
print(f"深掘りする月:{mlabel(jm)}({JUMP_MEASURE}・{JUMP_SIGNAL})、RMW18 = {fac['factor'].get(jm, np.nan) * 100:.2f}%")

coef_cell = {c: (1 / 6 if c.endswith("/H") else -1 / 6 if c.endswith("/L") else 0.0) for c in fac["rets"].columns}
cells = pd.DataFrame({"リターン(%)": fac["rets"].loc[jm] * 100, "銘柄数": fac["counts"].loc[jm],
                      "係数": pd.Series(coef_cell)})
cells["寄与(%)"] = cells["リターン(%)"] * cells["係数"]
cells = cells.sort_values("寄与(%)", key=lambda s: s.abs(), ascending=False)
display(cells)

# 最も寄与したセルの構成銘柄
top_cell = cells.index[0]
col = SIGNALS[JUMP_SIGNAL]
h = assign_groups(panels[JUMP_MEASURE].dropna(subset=["Y", "Yhat"]), col, bm_adjust=True)
fy_of_month = (jm - FORMATION_MONTH) // 12
mem = h[(h["form_year"] == fy_of_month) & (h["grp"] == top_cell)]
mem = mem.merge(mlong[mlong["midx"] == jm][["sec", "ret", "growth_before"]], on="sec", how="left")
mem["w"] = mem["ME_form"] * mem["growth_before"] if WEIGHTING == "vw" else mem["growth_before"]
mem["ウェイト(%)"] = mem["w"] / mem["w"].sum() * 100
mem["セル内の寄与(%)"] = mem["ウェイト(%)"] * mem["ret"]
mem["リターン(%)"] = mem["ret"] * 100
show_cols = ["NRI_CODE", "sec", "CTYPE", "ウェイト(%)", "リターン(%)", "セル内の寄与(%)", "Y", "Yhat", "lnBM", "ME_form"]
print(f"セル {top_cell}({fy_of_month} 年6月組成)の構成銘柄:{len(mem)} 銘柄(寄与の大きい順)")
display(mem.sort_values("セル内の寄与(%)", key=lambda s: s.abs(), ascending=False)[show_cols].head(15))

### 10-6. 寄与の大きい銘柄のリターンと時価総額の推移

寄与の大きい上位3銘柄について、ジャンプした月の前後6か月のリターンと月末時価総額を表示します。
1か月で数倍になってすぐ戻るなど、データの誤りを疑う動きがないか確認してください。

In [ ]:
top_secs = mem.sort_values("セル内の寄与(%)", key=lambda s: s.abs(), ascending=False)["sec"].head(3).tolist()
rng_m = range(jm - 6, jm + 7)
hist = {}
for sec in top_secs:
    r = ret[ret["sec"] == sec].set_index("midx")["ret"].reindex(rng_m) * 100
    me = mcap[mcap["sec"] == sec].set_index("midx")["me"].reindex(rng_m) / 1e8
    hist[(sec, "リターン(%)")] = r
    hist[(sec, "時価総額(億円)")] = me
history = pd.DataFrame(hist)
history.index = [mlabel(i) for i in history.index]
display(history)

### 10-7. 結果の読み方

| 見つかったこと | 原因 | 対処の候補 |
|---|---|---|
| 10-1 で、ある組成年に CP・ROE の「投資あり」の候補だけが極端に少ない | 自己資本成長率が計算できない(10-2 で、ある決算期の `g_BE` がほぼ 0 件) | 会社法施行の前後をまたぐ年は、項目が違っても自己資本(`BE`)の成長率を使う。または投資の変数を総資産成長率にそろえる |
| 10-3 で、その前後の月だけ銘柄数・最少セルが極端に少ない | サンプルが薄い年にセルが1〜2銘柄になる | 上の対処でサンプルを回復させる。最少セルの銘柄数に下限を設ける |
| 10-5・10-6 で、1銘柄の極端なリターンが寄与の大半を占める | 薄いセルでの個別銘柄の影響、またはリターンのデータの誤り | データの確認。リターンの外れ値の処理 |

結果を貼っていただければ、原因の特定と本分析の修正方法を一緒に決めます。

In [ ]:
OUTPUT_DIR = Path("outputs_jump_check")
OUTPUT_DIR.mkdir(exist_ok=True)
sample_by_year.to_csv(OUTPUT_DIR / "sample_by_year.csv", encoding="utf-8-sig")
equity_items.to_csv(OUTPUT_DIR / "equity_items_by_fiscal_year.csv", encoding="utf-8-sig")
window_table.to_csv(OUTPUT_DIR / "monthly_window.csv", encoding="utf-8-sig")
cells.to_csv(OUTPUT_DIR / f"cells_{JUMP_MEASURE}_{JUMP_SIGNAL}_{mlabel(jm)}.csv", encoding="utf-8-sig")
mem.to_csv(OUTPUT_DIR / f"members_{JUMP_MEASURE}_{JUMP_SIGNAL}_{mlabel(jm)}.csv", encoding="utf-8-sig", index=False)
history.to_csv(OUTPUT_DIR / f"history_{JUMP_MEASURE}_{JUMP_SIGNAL}_{mlabel(jm)}.csv", encoding="utf-8-sig")
print(f"保存先:{OUTPUT_DIR.resolve()}")